In [ ]:
!pip install doubleml

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from doubleml import DoubleMLData
from doubleml.plm import DoubleMLPLR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

np.random.seed(42)

## Configuração

Escolha a área alterando `AREA` (`'area1'` ou `'area2'`).

In [5]:
AREA = 'area1'  # ou 'area2'

CONFIG = {
    'area1': {
        'csv_path': '/content/dataarea1.csv',
        'onehot_cols': ['Mês', 'Dia_semana', 'Dia_mês', 'prato_principal_1',
                        'prato_principal_2', 'guarnição', 'sobremesa_1', 'refeicao'],
        # limpezas extras que só existiam no notebook do area1
        'remove_cafe_da_manha': True,
        'remove_erro_leitura': True,
        'remove_total_zero': True,
        'dropna': True,
        'image_dir': '../images/area1',
        'image_suffix': '',
    },
    'area2': {
        'csv_path': '../data/dataarea2.csv',
        'onehot_cols': ['Mês', 'Dia_semana', 'Dia_mês', 'prato_principal_1',
                        'prato_principal_2', 'guarnição', 'sobremesa_1'],
        'remove_cafe_da_manha': False,
        'remove_erro_leitura': False,
        'remove_total_zero': False,
        'dropna': False,
        'image_dir': '../images/area2',
        'image_suffix': '_2',
    },
}

cfg = CONFIG[AREA]

# Nome da coluna alvo (Y). Ver observação sobre 'total' x 'Total' na célula acima.
Y_COL = 'total'

# Também remove o dia 11/10/2023, igual nos dois notebooks originais
REMOVE_DIA_11_10_2023 = True

# Parâmetros do DoubleMLPLR
N_ESTIMATORS = 50
N_FOLDS = 3
RANDOM_STATE = 0

## Carregar e preparar os dados

In [ ]:
#Importar os dados
data = pd.read_csv(cfg['csv_path'], header=(0))  #arquivo gerado pelo notebook de agrupamento e limpeza

if cfg['remove_cafe_da_manha']:
    #Remover o Café da Manhã
    data = data[data['refeicao'] != 'cafe da manha'].copy()

if cfg['remove_erro_leitura']:
    #Remover erro de leitura
    data = data[data["guarnição"] != "Erro de leitura"]

if REMOVE_DIA_11_10_2023:
    #Remover dia 11/10/2023
    data = data[~(
        (data["Ano"] == 2023) &
        (data["Mês"] == 10) &
        (data["Dia_mês"] == 11)
    )]

if cfg['dropna']:
    data = data.dropna()

if cfg['remove_total_zero']:
    data = data[data[Y_COL] != 0]

#Descartar observações com valores que aparecem menos de 5 vezes
data = data.groupby("guarnição").filter(lambda x: len(x) >= 5)
data = data.groupby("prato_principal_1").filter(lambda x: len(x) >= 5)
data = data.groupby("prato_principal_2").filter(lambda x: len(x) >= 5)
data = data.groupby("sobremesa_1").filter(lambda x: len(x) >= 5)

#Aplicando One Hot Encoding nas variáveis categóricas
data = pd.get_dummies(data, columns=cfg['onehot_cols'])

data.shape

## Funções de Double Machine Learning

In [8]:
def run_dml_analysis(data, y_col, treatment_prefix,
                      n_estimators=N_ESTIMATORS, n_folds=N_FOLDS, random_state=RANDOM_STATE):
    """Roda o DoubleMLPLR para um grupo de variáveis dummy (D) que começam com `treatment_prefix`."""

    #Define D (Variável que queremos saber o impacto)
    d_cols = [c for c in data.columns if c.startswith(treatment_prefix)]

    #Define X (Variáveis de confusão)
    x_cols = [c for c in data.columns if c not in d_cols + [y_col]]

    #Criar o objeto do doubleml
    dml_data = DoubleMLData(
        data,
        y_col=y_col,
        d_cols=d_cols,
        x_cols=x_cols
    )

    #Escolher os modelos ML
    ml_m = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)  # modelo para D ~ X
    ml_l = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)   # modelo para Y ~ X

    #Rodar
    dml_plr = DoubleMLPLR(
        dml_data,
        ml_m=ml_m,
        ml_l=ml_l,
        n_folds=n_folds
    )
    dml_plr.fit()

    return d_cols, dml_plr


def plot_dml_coefficients(d_cols, coef, prefix_to_remove, title, save_path):
    """Gera o gráfico de barras com os 10 coeficientes mais positivos e mais negativos."""

    df_coef = pd.DataFrame({
        "Variável": d_cols,
        "Coeficiente": coef
    })

    # remover prefixo
    df_coef["Variável"] = df_coef["Variável"].str.replace(prefix_to_remove, "", regex=False)

    # Ordenar do menor para o maior
    df_sorted = df_coef.sort_values(by="Coeficiente")

    # 10 menores (mais negativos) e 10 maiores (mais positivos)
    bottom10 = df_sorted.head(10)
    top10 = df_sorted.tail(10)

    # Juntar para plotar
    plot_df = pd.concat([bottom10, top10])

    plt.figure()
    plt.barh(plot_df["Variável"], plot_df["Coeficiente"])
    plt.axvline(0)  # linha no zero para referência
    plt.xlabel("Coeficiente")
    plt.title(title)
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()

## Executar para os quatro grupos

Prato principal 1, prato principal 2, guarnição e sobremesa

In [ ]:
import os

# prefixo da coluna dummy -> (nome do arquivo de imagem, título do gráfico)
TREATMENT_GROUPS = {
    'prato_principal_1_': ('sppp1', 'Score de Popularidade do prato principal 1'),
    'prato_principal_2_': ('sppp2', 'Score de Popularidade do prato principal 2'),
    'guarnição_':          ('spg',   'Score de Popularidade da guarnição'),
    'sobremesa_1_':       ('sps',   'Score de Popularidade das sobremesas'),
}

results = {}

for prefix, (img_name, title) in TREATMENT_GROUPS.items():
    print(f'--- Rodando DML: {title} ---')

    d_cols, dml_plr = run_dml_analysis(data, y_col=Y_COL, treatment_prefix=prefix)
    results[prefix] = (d_cols, dml_plr)

    #Resultados
    for nome, valor in sorted(zip(d_cols, dml_plr.coef), key=lambda x: x[1], reverse=True):
        print(nome, "→", valor)

    # Ensure the directory exists before saving the image
    os.makedirs(cfg['image_dir'], exist_ok=True)
    save_path = f"{cfg['image_dir']}/{img_name}{cfg['image_suffix']}.png"
    plot_dml_coefficients(d_cols, dml_plr.coef, prefix, title, save_path)

    print()